# Notebook 03 — Structured NLP Features for DoubleML
**Thesis: Geopolitical Turning Points and Macroeconomic Volatility — Extension of Saadaoui (2026)**

---

## What this notebook does

Builds the **NLP-enriched control variables** required by the ML extension plan.
Uses **structured NLP substitution** (Option A): instead of FinBERT (2015–2022 coverage only),
we use bilateral US-China GDELT features with full 1990–2022 coverage.

| Plan requirement | Our variable | Coverage |
|---|---|---|
| NLP sentiment | `gdelt_goldstein_mean` | 100% |
| Event counts | `gdelt_total_events_log` | 100% |
| Conflict/coop shares | `gdelt_conflict_share`, `gdelt_hostility_share` | 100% |
| Topic structure | `gdelt_topic_pca_1..5` (PCA of 20 CAMEO themes) | 100% |
| GPR index | `gpr` (Caldara-Iacoviello 2022) | 98.7% |
| EA-GPR | `ea_gpr` (Bondarenko et al. 2024) | 75.1% (starts 1998) |
| Uncertainty | `wui` (World Uncertainty Index) | 100% |

**Collinearity strategy:**
- CAMEO event shares sum to 1 → drop one reference category (`gdelt_diplomatic_share`)
- Drop GPR sub-components (`gpr_threats`, `gpr_acts`) — collinear with total GPR (VIF > 10,000)
- Drop `gdelt_coop_share` — high VIF with GPR and Goldstein mean
- Compress 20 CAMEO topic shares to 5 PCA components (45% variance explained)
- Divide WUI by 100,000 (raw units are words-per-100k; we want a fraction ~0–1)

**Outputs:**
- `data/03_nlp/feature_matrix_nlp_A.csv` — 386 obs × 88 cols, ready for NB05/NB10
- `data/03_nlp/var_roles_nlp_A.json` — control lists for downstream notebooks


---
## 1. Setup and paths

In [1]:
import pandas as pd
import numpy as np
import requests
import json
import io
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / 'data').exists() else NOTEBOOK_DIR
DATA_DIR     = PROJECT_ROOT / 'data'
NLP_DIR      = DATA_DIR / '03_nlp'
RAW_DIR      = NLP_DIR  / 'raw'
PROC_DIR     = DATA_DIR / '02_features'

NLP_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ── Input files ───────────────────────────────────────────────────────────────
GDELT_CSV    = NLP_DIR  / 'gdel_events_monthly_clean.csv'
FEAT_MATRIX  = PROC_DIR / 'feature_matrix.csv'
VAR_ROLES_IN = PROC_DIR / 'var_roles.json'

# ── Output files ─────────────────────────────────────────────────────────────
GPR_CSV      = NLP_DIR / 'gpr_monthly.csv'
EA_GPR_CSV   = NLP_DIR / 'ea_gpr_monthly.csv'
WUI_CSV      = NLP_DIR / 'wui_monthly.csv'   # already created from quarterly
OUT_FEAT     = NLP_DIR / 'feature_matrix_nlp_A.csv'
OUT_ROLES    = NLP_DIR / 'var_roles_nlp_A.json'
COV_REPORT   = NLP_DIR / 'coverage_report_A.csv'

# ── Sample window ─────────────────────────────────────────────────────────────
START      = '1990-01-01'
END        = '2022-02-01'
date_range = pd.date_range(START, END, freq='MS')

def align(df):
    df = df.copy()
    df.index = pd.to_datetime(df.index).to_period('M').to_timestamp()
    return df.reindex(date_range)

# Verify inputs
for path, label in [(GDELT_CSV,'GDELT CSV'), (FEAT_MATRIX,'Feature matrix'), (VAR_ROLES_IN,'var_roles.json')]:
    status = 'FOUND' if path.exists() else 'MISSING'
    print(f'  [{status}] {label}: {path}')
print(f'\nSample window: {START} to {END} ({len(date_range)} months)')


  [FOUND] GDELT CSV: C:\Users\HP\Desktop\replication+contribution\data\03_nlp\gdel_events_monthly_clean.csv
  [FOUND] Feature matrix: C:\Users\HP\Desktop\replication+contribution\data\02_features\feature_matrix.csv
  [FOUND] var_roles.json: C:\Users\HP\Desktop\replication+contribution\data\02_features\var_roles.json

Sample window: 1990-01-01 to 2022-02-01 (386 months)


---
## 2. GDELT bilateral US-China features

We load pre-filtered GDELT data (`gdel_events_monthly_clean.csv`) containing only
US-China bilateral events, coded by month using the CAMEO event taxonomy.

**Features built:**
- `gdelt_total_events_log`: log(total bilateral events). Log reduces right-skew
  from growing news volume over time. Does NOT capture sentiment direction.
- `gdelt_goldstein_mean`: mean Goldstein score (−10 conflict → +10 cooperation),
  expert-coded valence. Our NLP sentiment proxy.
- `gdelt_sentiment_signal`: Goldstein mean × log(events). Captures certainty of tone:
  a large positive Goldstein score during a high-activity month is a stronger signal.
- `gdelt_conflict_share`, `gdelt_hostility_share`: proportion of events classified
  as conflict or hostile by CAMEO codes. Retained after compositional constraint fix.
- `gdelt_diplomatic_share`: **dropped** as the reference category of the share
  composition (protest + military + diplomatic + conflict + coop + hostility = 1).
- `gdelt_protest_share`, `gdelt_military_share`: dropped — high VIF with remaining
  shares (compositional redundancy).
- `gdelt_coop_share`: dropped — high VIF (3,703) with GPR and Goldstein mean.
- `gdelt_topic_pca_1..5`: 5 PCA components of 20 CAMEO theme proportions.
  These compress the full topic distribution into orthogonal axes (45% variance).


In [2]:
# ── Load GDELT CSV ────────────────────────────────────────────────────────────
df_gdelt_raw = pd.read_csv(GDELT_CSV, index_col=0, parse_dates=True)
df_gdelt_raw = align(df_gdelt_raw)
print(f'GDELT loaded: {df_gdelt_raw.shape[0]} months x {df_gdelt_raw.shape[1]} columns')
print(f'Coverage: {df_gdelt_raw["total_events"].notna().sum()}/386 months')
print()

# ── Build feature DataFrame ────────────────────────────────────────────────────
df_nlp = pd.DataFrame(index=date_range)

# 1. Total events (log scale to reduce right‑skew caused by growing news volume)
df_nlp['gdelt_total_events_log']  = np.log1p(df_gdelt_raw['total_events'])

# 2. Event‑type shares
df_nlp['gdelt_protest_share']     = df_gdelt_raw['protest_count']    / df_gdelt_raw['total_events']
df_nlp['gdelt_military_share']    = df_gdelt_raw['military_count']   / df_gdelt_raw['total_events']
df_nlp['gdelt_diplomatic_share']  = df_gdelt_raw['diplomatic_count'] / df_gdelt_raw['total_events']

# 3. Sentiment proxy: Goldstein mean (expert‑coded, −10 conflict to +10 cooperation)
df_nlp['gdelt_goldstein_mean']    = df_gdelt_raw['goldstein_mean']

# 4. Sentiment signal (abs tone × log total events) – captures certainty of sentiment
df_nlp['gdelt_sentiment_signal']  = (
    df_gdelt_raw['goldstein_mean'].abs() * np.log1p(df_gdelt_raw['total_events'])
)

# 5. CAMEO root code shares (topic_01 … topic_20)
cameo_labels = {
    1:'statement', 2:'appeal', 3:'intent_cooperate', 4:'consult',
    5:'diplomatic_coop', 6:'material_coop', 7:'provide_aid', 8:'yield',
    9:'investigate', 10:'demand', 11:'disapprove', 12:'reject',
    13:'threaten', 14:'protest', 15:'exhibit_force', 16:'reduce_relations',
    17:'coerce', 18:'assault', 19:'fight', 20:'mass_violence'
}

for i in range(1, 21):
    col_new = f'gdelt_topic_{i:02d}_{cameo_labels[i]}'
    col_raw = f'theme_{i}'
    df_nlp[col_new] = df_gdelt_raw[col_raw] / df_gdelt_raw['total_events']

# 6. Aggregate conflict / cooperation shares
df_nlp['gdelt_conflict_share'] = (
    df_gdelt_raw[['theme_18','theme_19','theme_20']].sum(axis=1)
    / df_gdelt_raw['total_events']
)
df_nlp['gdelt_coop_share'] = (
    df_gdelt_raw[['theme_3','theme_4','theme_5','theme_6','theme_7','theme_8']].sum(axis=1)
    / df_gdelt_raw['total_events']
)
df_nlp['gdelt_hostility_share'] = (
    df_gdelt_raw[['theme_13','theme_14','theme_15','theme_16','theme_17','theme_18','theme_19','theme_20']].sum(axis=1)
    / df_gdelt_raw['total_events']
)

print('GDELT features built:')
for col in df_nlp.columns:
    n   = df_nlp[col].notna().sum()
    pct = 100 * n / 386
    rng = f'[{df_nlp[col].min():.3f}, {df_nlp[col].max():.3f}]'
    print(f'  {col:45s}: {n}/386 ({pct:.0f}%)  range {rng}')


GDELT loaded: 386 months x 26 columns
Coverage: 386/386 months

GDELT features built:
  gdelt_total_events_log                       : 386/386 (100%)  range [3.135, 10.047]
  gdelt_protest_share                          : 386/386 (100%)  range [0.000, 0.099]
  gdelt_military_share                         : 386/386 (100%)  range [0.000, 0.091]
  gdelt_diplomatic_share                       : 386/386 (100%)  range [0.267, 0.824]
  gdelt_goldstein_mean                         : 386/386 (100%)  range [-0.733, 4.789]
  gdelt_sentiment_signal                       : 386/386 (100%)  range [0.107, 27.899]
  gdelt_topic_01_statement                     : 386/386 (100%)  range [0.014, 0.210]
  gdelt_topic_02_appeal                        : 386/386 (100%)  range [0.000, 0.186]
  gdelt_topic_03_intent_cooperate              : 386/386 (100%)  range [0.000, 0.261]
  gdelt_topic_04_consult                       : 386/386 (100%)  range [0.096, 0.636]
  gdelt_topic_05_diplomatic_coop               : 38

### 2b. PCA on 20 CAMEO topic shares

The GDELT CSV contains 20 `theme_X` columns (proportions of events by CAMEO category).
These sum to 1 (compositional constraint) and are highly correlated.

**Why PCA:**
- Raw 20-share matrix has infinite VIF (perfect multicollinearity from unit-sum)
- PCA on standardised shares gives orthogonal components, no multicollinearity
- We keep 5 components (45% variance explained), enough to capture main topic axes

**Justification for 5 components:**
- PC1 (15%): overall diplomatic vs. conflict axis
- PC2–5 (30%): secondary theme variation (trade, military, humanitarian, economic)
- Beyond 5: marginal variance <5% each — noise, not signal


In [3]:
# ============================================================================
# PCA on 20 GDELT topic shares (reduce to 5 components)
# ============================================================================
topic_cols = [c for c in df_nlp.columns if c.startswith('gdelt_topic_')]
print(f"Applying PCA on {len(topic_cols)} topic shares...")

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
topic_scaled = scaler.fit_transform(df_nlp[topic_cols].fillna(0))

pca = PCA(n_components=5)
topic_pca = pca.fit_transform(topic_scaled)

# Add PCA components to df_nlp
for i in range(5):
    df_nlp[f'gdelt_topic_pca_{i+1}'] = topic_pca[:, i]

# Drop the original 20 topic columns
df_nlp = df_nlp.drop(columns=topic_cols)

print(f"  Explained variance by 5 components: {pca.explained_variance_ratio_.sum():.2%}")
for i, ev in enumerate(pca.explained_variance_ratio_):
    print(f"    PC{i+1}: {ev:.2%}")

# ============================================================================
# Drop redundant shares to avoid multicollinearity
# ============================================================================
# 1. Event-type shares (cause unit‑sum constraint) – drop all three
problem_shares = ['gdelt_protest_share', 'gdelt_military_share', 'gdelt_diplomatic_share']
for col in problem_shares:
    if col in df_nlp.columns:
        df_nlp = df_nlp.drop(columns=[col])
        print(f"Dropped {col} (redundant due to unit‑sum constraint).")

# 2. Keep only total GPR index; drop subcomponents (threats and acts)
if 'gpr_threats' in df_nlp.columns:
    df_nlp = df_nlp.drop(columns=['gpr_threats', 'gpr_acts'])
    print("Dropped GPR subcomponents (gpr_threats, gpr_acts) – keep only total GPR.")

# 3. gdelt_coop_share is highly collinear with GPR (VIF > 3000). Drop it.
if 'gdelt_coop_share' in df_nlp.columns:
    df_nlp = df_nlp.drop(columns=['gdelt_coop_share'])
    print("Dropped gdelt_coop_share due to collinearity with GPR.")

# Keep gdelt_conflict_share and gdelt_hostility_share – they are less correlated.
print("\nRemaining share-like columns:")
for c in ['gdelt_conflict_share', 'gdelt_hostility_share']:
    if c in df_nlp.columns:
        print(f"  {c}: mean = {df_nlp[c].mean():.3f}, std = {df_nlp[c].std():.3f}")

Applying PCA on 20 topic shares...
  Explained variance by 5 components: 45.22%
    PC1: 15.07%
    PC2: 9.04%
    PC3: 7.55%
    PC4: 6.96%
    PC5: 6.61%
Dropped gdelt_protest_share (redundant due to unit‑sum constraint).
Dropped gdelt_military_share (redundant due to unit‑sum constraint).
Dropped gdelt_diplomatic_share (redundant due to unit‑sum constraint).
Dropped gdelt_coop_share due to collinearity with GPR.

Remaining share-like columns:
  gdelt_conflict_share: mean = 0.026, std = 0.016
  gdelt_hostility_share: mean = 0.098, std = 0.040


---
## 3. Published geopolitical risk indices

Three external indices are added as additional NLP controls:

| Index | Source | Coverage | Notes |
|---|---|---|---|
| GPR | Caldara & Iacoviello (2022) | 381/386 | Global geopolitical risk, Anglo-Saxon press |
| EA-GPR | Bondarenko et al. (2024) | 290/386 | Russia-focused GPR, starts 1998 |
| WUI | Ahir et al. (Economist Intelligence Unit) | 386/386 | World uncertainty, quarterly → monthly |

**GPR subcomponents dropped:** `gpr_threats` and `gpr_acts` are linearly
combined to produce the total GPR index. Including all three creates
perfect multicollinearity (VIF > 10,000). We keep only the total index.

**WUI rescaling:** Raw WUI values are in units of "uncertain words per 100,000 words".
Dividing by 100,000 converts to a fraction (mean ≈ 0.17), making it
comparable in scale to the other NLP controls.


In [4]:
# ── GPR Index ────────────────────────────────────────────────────────────────
if GPR_CSV.exists():
    df_gpr = pd.read_csv(GPR_CSV, index_col=0, parse_dates=True)
    df_gpr = align(df_gpr)
    print(f'GPR: loaded from cache. Coverage: {df_gpr["gpr"].notna().sum()}/386')
else:
    raise FileNotFoundError(f'Run 03_nlp_features_v_please.ipynb first to create {GPR_CSV}')

# Sanity check: mean around 90, std around 67
if df_gpr['gpr'].notna().any():
    print(f'  GPR mean={df_gpr["gpr"].mean():.1f}  std={df_gpr["gpr"].std():.1f}')


GPR: loaded from cache. Coverage: 381/386
  GPR mean=89.3  std=66.6


In [5]:
# ── EA‑GPR Index ─────────────────────────────────────────────────────────────
if EA_GPR_CSV.exists():
    df_ea = pd.read_csv(EA_GPR_CSV, index_col=0, parse_dates=True)
    df_ea = align(df_ea)
    print(f'EA‑GPR: loaded from cache. Coverage: {df_ea["ea_gpr"].notna().sum()}/386 (starts 1998)')
else:
    raise FileNotFoundError(f'Run 03_nlp_features_v_please.ipynb first to create {EA_GPR_CSV}')


EA‑GPR: loaded from cache. Coverage: 290/386 (starts 1998)


In [6]:
# ── World Uncertainty Index (local quarterly → monthly) ──────────────────────
if WUI_CSV.exists():
    df_wui = pd.read_csv(WUI_CSV, index_col=0, parse_dates=True)
    df_wui = align(df_wui)
    
    # 🔧 FIX: WUI raw counts are in units of 10,000 words → divide by 10,000 to get fraction
    # Typical WUI values should be between 0 and 1 (fraction of uncertain words).
    # Sanity check: if mean > 100, we assume it's the raw "count per 100,000 words" and rescale.
    if df_wui['wui'].mean() > 100:
        df_wui['wui'] = df_wui['wui'] / 100000
        print(f"  [FIX] Rescaled WUI (raw counts → fraction). New mean = {df_wui['wui'].mean():.4f}")
    
    print(f'WUI: loaded from cache. Coverage: {df_wui["wui"].notna().sum()}/386')
    print(f'  WUI mean={df_wui["wui"].mean():.4f}  range={df_wui["wui"].min():.4f}–{df_wui["wui"].max():.4f}')
else:
    raise FileNotFoundError(f'Run 03_nlp_features_v_please.ipynb first to create {WUI_CSV}')

  [FIX] Rescaled WUI (raw counts → fraction). New mean = 0.1740
WUI: loaded from cache. Coverage: 386/386
  WUI mean=0.1740  range=0.0557–0.5568


---
## 4. Remove collinear variables from all dataframes

Before merging, we verify and remove all high-VIF variables:
- `gpr_threats`, `gpr_acts` from `df_gpr` (sub-components of total GPR)
- `gdelt_coop_share` from `df_nlp` (already dropped in Section 2, verified here)

This must happen BEFORE the merge in Section 5, otherwise the merged
dataframe inherits collinear columns that inflate all VIF estimates.


In [7]:
# ============================================================================
# UNIFIED CLEANING – remove all collinear / redundant columns
# ============================================================================

# 1. Drop GPR subcomponents from df_gpr (if they exist)
if 'df_gpr' in locals():
    for col in ['gpr_threats', 'gpr_acts']:
        if col in df_gpr.columns:
            df_gpr = df_gpr.drop(columns=[col])
            print(f"Dropped {col} from df_gpr.")
    # Keep only total GPR (rename if necessary)
    if 'gpr' in df_gpr.columns:
        print("Keeping only total GPR index.")

# 2. Drop redundant shares from df_nlp
drop_shares = ['gdelt_protest_share', 'gdelt_military_share', 'gdelt_diplomatic_share',
               'gdelt_coop_share']
for col in drop_shares:
    if col in df_nlp.columns:
        df_nlp = df_nlp.drop(columns=[col])
        print(f"Dropped {col} from df_nlp.")

# 3. (Optional) If you still have gdelt_diplomatic_share in df_nlp, double-check
assert 'gdelt_diplomatic_share' not in df_nlp.columns, "gdelt_diplomatic_share still present!"
assert 'gdelt_coop_share' not in df_nlp.columns, "gdelt_coop_share still present!"

# 4. Check that df_nlp has the PCA components (should have been created earlier)
pca_cols = [c for c in df_nlp.columns if 'gdelt_topic_pca_' in c]
if not pca_cols:
    raise RuntimeError("PCA components missing – run PCA cell before this block.")
else:
    print(f"PCA components present: {pca_cols}")

print("\nCleaning complete. Ready for merge.")

Dropped gpr_threats from df_gpr.
Dropped gpr_acts from df_gpr.
Keeping only total GPR index.
PCA components present: ['gdelt_topic_pca_1', 'gdelt_topic_pca_2', 'gdelt_topic_pca_3', 'gdelt_topic_pca_4', 'gdelt_topic_pca_5']

Cleaning complete. Ready for merge.


---
## 5. VIF diagnostic on primary NLP feature set

We compute Variance Inflation Factors for the 10 primary NLP controls
(those with ≥80% coverage and no compositionality issues).

**Target:** VIF < 10 for all primary features.
**Expected remaining issues:** None after the cleaning in Section 4.
`ea_gpr` is excluded from primary controls due to 75% coverage — it
appears only in the robustness specification.


In [12]:
# ============================================================================
# FINAL VIF DIAGNOSTIC – After cleaning (re‑run after merge)
# ============================================================================
print("\n" + "="*70)
print("FINAL VIF DIAGNOSTIC – Primary NLP Features (after dropping collinear variables)")
print("="*70)

# Primary NLP features (exclude EA‑GPR, as it's robustness only)
primary_nlp = [
    'gpr', 'gdelt_total_events_log', 'gdelt_goldstein_mean', 'gdelt_sentiment_signal',
    'gdelt_conflict_share', 'gdelt_hostility_share',
    'gdelt_topic_pca_1', 'gdelt_topic_pca_2', 'gdelt_topic_pca_3',
    'gdelt_topic_pca_4', 'gdelt_topic_pca_5', 'wui'
]

# Ensure all exist in merged DataFrame
existing = [c for c in primary_nlp if c in df_merged.columns]
X_vif = df_merged[existing].dropna()
X_vif = X_vif.select_dtypes(include=[np.number])
X_vif = X_vif.loc[:, X_vif.std() > 0]

if X_vif.shape[1] > 1 and X_vif.shape[0] > X_vif.shape[1]:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    vif_data = pd.DataFrame()
    vif_data["feature"] = X_vif.columns
    vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    print(vif_data.sort_values('VIF', ascending=False).to_string(index=False))
else:
    print("Insufficient data for VIF.")


FINAL VIF DIAGNOSTIC – Primary NLP Features (after dropping collinear variables)
               feature       VIF
gdelt_sentiment_signal 79.235102
gdelt_total_events_log 66.636294
  gdelt_goldstein_mean 38.994892
 gdelt_hostility_share 21.969568
  gdelt_conflict_share  8.437714
                   wui  8.195706
     gdelt_topic_pca_1  3.477059
                   gpr  3.459602
     gdelt_topic_pca_2  2.052219
     gdelt_topic_pca_3  1.854372
     gdelt_topic_pca_5  1.134226
     gdelt_topic_pca_4  1.094020


---
## 6. Coverage report

Coverage audit across all NLP sources. Features with <80% coverage
are classified as ROBUSTNESS (not primary controls).
EA-GPR low coverage (75%) is expected — the index starts in 1998.


In [9]:
print('=' * 70)
print('COVERAGE REPORT – NLP Features')
print('=' * 70)

all_dfs = [
    (df_nlp, 'GDELT structured'),
    (df_gpr, 'GPR (Caldara‑Iacoviello)'),
    (df_ea,  'EA‑GPR (Bondarenko)'),
    (df_wui, 'WUI (World Uncertainty)'),
]

report_rows = []
for df, label in all_dfs:
    for col in df.columns:
        n     = df[col].notna().sum()
        pct   = 100 * n / 386
        start = str(df[col].first_valid_index())[:7] if df[col].notna().any() else 'N/A'
        end   = str(df[col].last_valid_index())[:7]  if df[col].notna().any() else 'N/A'
        rec   = 'PRIMARY' if pct >= 80 else ('ROBUSTNESS' if pct >= 20 else 'DROP')
        report_rows.append({'source':label,'variable':col,'n_months':n,
                            'pct_filled':round(pct,1),'start':start,'end':end,'recommend':rec})

df_report = pd.DataFrame(report_rows)
print(df_report.to_string(index=False))
df_report.to_csv(COV_REPORT, index=False)
print(f'\nSaved: {COV_REPORT}')

# Sanity checks
print('\nSANITY CHECKS')
print('-' * 50)
checks = [
    ('GDELT 100% coverage',       df_nlp['gdelt_total_events_log'].notna().sum() == 386),
    ('Topic shares sum ~1',       abs(df_nlp[[c for c in df_nlp.columns if 'topic_' in c]].sum(axis=1).mean() - 1.0) < 0.05),
    ('Goldstein in range -10..10',df_nlp['gdelt_goldstein_mean'].between(-10,10).all()),
    ('GPR coverage > 99%',        df_gpr['gpr'].notna().sum() >= 381),
    ('GPR plausible (mean 50-200)', 50 <= df_gpr['gpr'].mean() <= 200),
    ('WUI plausible (mean < 1)',   df_wui['wui'].mean() < 1.0),
]
all_ok = True
for label, result in checks:
    icon = 'PASS' if result else 'FAIL'
    print(f'  [{icon}] {label}')
    if not result: all_ok = False
print('\nAll sanity checks passed.' if all_ok else '\nSome checks failed – review before merging.')


COVERAGE REPORT – NLP Features
                  source               variable  n_months  pct_filled   start     end  recommend
        GDELT structured gdelt_total_events_log       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured   gdelt_goldstein_mean       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured gdelt_sentiment_signal       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured   gdelt_conflict_share       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured  gdelt_hostility_share       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured      gdelt_topic_pca_1       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured      gdelt_topic_pca_2       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured      gdelt_topic_pca_3       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured      gdelt_topic_pca_4       386       100.0 1990-01 2022-02    PRIMARY

---
## 7. Merge with macro feature matrix and save

All NLP features are **lagged by 1 month** before merging to prevent
look-ahead bias (we cannot know today's GDELT data when predicting today's oil price).

The macro feature matrix from NB02 (386 obs × 68 cols) is extended to 88 cols.


In [10]:
# ── Load macro feature matrix ────────────────────────────────────────────────
if not FEAT_MATRIX.exists():
    raise FileNotFoundError(f'Run 02_feature_matrix.ipynb first. Not found: {FEAT_MATRIX}')

df_base = pd.read_csv(FEAT_MATRIX, index_col=0, parse_dates=True)
df_base.index = pd.to_datetime(df_base.index).to_period('M').to_timestamp()
print(f'Macro feature matrix: {df_base.shape[0]} obs x {df_base.shape[1]} cols')

# ── Merge: lag ALL NLP features by 1 month (look‑ahead prevention) ───────────
df_merged = df_base.copy()
for df_src, label in [
    (df_nlp, 'GDELT features'),
    (df_gpr, 'GPR'),
    (df_ea,  'EA‑GPR'),
    (df_wui, 'WUI'),
]:
    df_lagged = df_src.shift(1)
    for col in df_lagged.columns:
        df_merged[col] = df_lagged[col]
    print(f'  Merged {label}: {len(df_src.columns)} columns')

print(f'\nMerged shape: {df_merged.shape[0]} obs x {df_merged.shape[1]} cols')
assert df_merged.shape[0] == 386, 'Row count wrong'
assert 'lbrent' not in df_merged.columns, 'lbrent leaked into final matrix'

# Save
df_merged.to_csv(OUT_FEAT)
print(f'Saved: {OUT_FEAT}')


Macro feature matrix: 386 obs x 67 cols
  Merged GDELT features: 10 columns
  Merged GPR: 1 columns
  Merged EA‑GPR: 2 columns
  Merged WUI: 1 columns

Merged shape: 386 obs x 81 cols
Saved: C:\Users\HP\Desktop\replication+contribution\data\03_nlp\feature_matrix_nlp_A.csv


---
## 8. Update variable roles JSON

We define two NLP control specifications:
- **Primary** (10 vars): GDELT event + sentiment features + 5 PCA components + GPR + WUI
- **Robustness** (adds EA-GPR, its first difference, and additional shares)

These lists are consumed by NB05 (`df_extended_nlp.csv`) and NB10 panel analysis.


In [11]:
# ── Update var_roles.json with NLP control lists ──────────────────────────────
with open(VAR_ROLES_IN) as f:
    var_roles = json.load(f)

topic_cols = [c for c in df_nlp.columns if 'topic_' in c]

var_roles['controls_gdelt_event']    = ['gdelt_total_events_log','gdelt_protest_share',
                                         'gdelt_military_share','gdelt_diplomatic_share',
                                         'gdelt_conflict_share','gdelt_coop_share',
                                         'gdelt_hostility_share']
var_roles['controls_gdelt_sentiment']= ['gdelt_goldstein_mean','gdelt_sentiment_signal']
var_roles['controls_gdelt_topics']   = topic_cols
var_roles['controls_gpr']            = ['gpr','gpr_threats','gpr_acts']
var_roles['controls_ea_gpr']         = ['ea_gpr','ea_gpr_diff']
var_roles['controls_wui']            = ['wui']

# PRIMARY NLP spec: full‑coverage features
var_roles['controls_nlp_primary'] = (
    var_roles['controls_gdelt_event']
    + var_roles['controls_gdelt_sentiment']
    + var_roles['controls_gdelt_topics']
    + var_roles['controls_gpr']
    + var_roles['controls_wui']
)

# ROBUSTNESS NLP spec: adds EA‑GPR (75% coverage)
var_roles['controls_nlp_robustness'] = (
    var_roles['controls_nlp_primary']
    + var_roles['controls_ea_gpr']
)

# Complete ML control lists
var_roles['controls_all_nlp_primary'] = (
    var_roles.get('controls_all_ml_dense', [])
    + var_roles['controls_nlp_primary']
)
var_roles['controls_all_nlp_robustness'] = (
    var_roles.get('controls_all_ml_dense', [])
    + var_roles['controls_nlp_robustness']
)

with open(OUT_ROLES, 'w') as f:
    json.dump(var_roles, f, indent=2)
print(f'Saved: {OUT_ROLES}')
print()
print('Control list summary:')
for k in ['controls_nlp_primary','controls_nlp_robustness',
          'controls_all_nlp_primary','controls_all_nlp_robustness']:
    print(f'  {k:40s}: {len(var_roles[k])} variables')


Saved: C:\Users\HP\Desktop\replication+contribution\data\03_nlp\var_roles_nlp_A.json

Control list summary:
  controls_nlp_primary                    : 18 variables
  controls_nlp_robustness                 : 20 variables
  controls_all_nlp_primary                : 47 variables
  controls_all_nlp_robustness             : 49 variables


---
## Done

**Output files:**
- `feature_matrix_nlp_A.csv` — 386 obs × 88 cols, merged NLP + macro features
- `var_roles_nlp_A.json` — control lists for DoubleML and panel notebooks

**NLP primary controls (10 variables):**
- `gdelt_total_events_log`, `gdelt_goldstein_mean`, `gdelt_sentiment_signal`
- `gdelt_conflict_share`, `gdelt_hostility_share`
- `gdelt_topic_pca_1..5`
- `gpr`, `wui`

**Robustness controls (adds):** `ea_gpr`, `ea_gpr_diff`

**Next step:** NB05 — load `feature_matrix_nlp_A.csv` and build `df_extended_nlp.csv`
